In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import time
import logging
import pandas as pd
from openai import AzureOpenAI
from rdflib import Graph, Namespace
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# --- Configure Logging for Verbose Output ---
# This sets up a logger to print detailed information about the script's progress.
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# --- API Constants and Configuration ---
# Load from environment variables (set in .env file)
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
MODEL_NAME = os.getenv("AZURE_OPENAI_MODEL_NAME", "o3-mini-1")

# --- File and Batching Configuration ---
INPUT_FILE_PATH = 'NVD110_VS2_20250709.txt'
OUTPUT_FILE_PATH = 'SMART_analysis_results.xlsx'

# --- FIX: Reduced BATCH_SIZE to prevent exceeding the token limit ---
# A batch size of 15 was too large for the max_completion_tokens limit.
# 5 is a proven safe value for this prompt and model.
BATCH_SIZE = 5

# Set to a number (e.g., 200) to only process that many requirements.
# Set to None to process all requirements in the file.
MAX_REQUIREMENTS_TO_ANALYZE = 50

# The maximum number of tokens the model can generate in the response.
# 4000 was not enough for a batch of 15, but it is sufficient for a batch of 5.
MAX_COMPLETION_TOKENS = 10000 

# --- Initialize Azure OpenAI Client ---
try:
    client = AzureOpenAI(
        azure_endpoint=AZURE_ENDPOINT,
        api_key=API_KEY,
        api_version=API_VERSION,
    )
    logging.info("Azure OpenAI client initialized successfully.")
except Exception as e:
    logging.error(f"Failed to initialize Azure OpenAI client: {e}")
    client = None

In [2]:
# def extract_requirements_from_file(file_path: str) -> list:
#     """
#     Parses a Turtle (.txt) file to extract requirements using rdflib.

#     Args:
#         file_path: The path to the input .txt file.

#     Returns:
#         A list of dictionaries, where each dictionary represents a requirement.
#     """
#     if not os.path.exists(file_path):
#         logging.error(f"Input file '{file_path}' not found.")
#         return []

#     logging.info(f"Starting requirement extraction from: {file_path}")
    
#     graph = Graph()
#     try:
#         # Parse the file, specifying the format is 'turtle'
#         graph.parse(file_path, format="turtle")
#     except Exception as e:
#         logging.error(f"Failed to parse the Turtle file: {e}")
#         return []

#     # SPARQL query to find all subjects that are of type nen2660:Requirement
#     # and get their associated notation, label, and value.
#     query = """
#         SELECT ?subject ?notation ?prefLabel ?value
#         WHERE {
#             ?subject a <https://w3id.org/nen2660/def#Requirement> .
#             OPTIONAL { ?subject <http://www.w3.org/2004/02/skos/core#notation> ?notation . }
#             OPTIONAL { ?subject <http://www.w3.org/2004/02/skos/core#prefLabel> ?prefLabel . }
#             OPTIONAL { ?subject <http://www.w3.org/1999/02/22-rdf-syntax-ns#value> ?value . }
#         }
#     """
    
#     results = graph.query(query)
    
#     requirements_list = []
#     for row in results:
#         req_text = str(row.value) if row.value else ""
#         # Only add the requirement if it has actual text content
#         if req_text.strip():
#             requirements_list.append({
#                 'id': str(row.notation) if row.notation else "N/A",
#                 'label': str(row.prefLabel) if row.prefLabel else "N/A",
#                 'text': req_text
#             })

#     logging.info(f"Successfully extracted {len(requirements_list)} requirements.")
#     return requirements_list

In [3]:
def extract_requirements_from_file(file_path: str) -> list:
    """
    PHASE 1: THE RDF PARSER (ENHANCED VERSION).
    This function now parses the TTL file and identifies parent-child relationships
    between requirements. It enriches each child requirement with the text of its
    parent to provide full context for the LLM analysis.

    Args:
        file_path: The path to the input .txt file.

    Returns:
        A list of dictionaries, where each dictionary represents a requirement
        and includes contextual parent information if available.
    """
    if not os.path.exists(file_path):
        logging.error(f"Input file '{file_path}' not found.")
        return []

    logging.info(f"Starting contextual requirement extraction from: {file_path}")
    
    graph = Graph()
    try:
        graph.parse(file_path, format="turtle")
    except Exception as e:
        logging.error(f"Failed to parse the Turtle file: {e}")
        return []

    # This SPARQL query is more advanced. It finds all requirements and,
    # for each one, it OPTIONALLY looks for a "parent" requirement that
    # it is derived from using the 'isDerivedIn' property.
    query = """
        PREFIX nen2660: <https://w3id.org/nen2660/def#>
        PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX tennet: <http://data.tennet.eu/def/>

        SELECT ?req ?notation ?label ?value ?parent_notation ?parent_value
        WHERE {
            ?req a nen2660:Requirement .
            ?req skos:notation ?notation .
            ?req rdf:value ?value .
            OPTIONAL { ?req skos:prefLabel ?label . }
            
            # Optional: Find a parent that this requirement is derived from
            OPTIONAL {
                ?parent tennet:isDerivedIn ?req .
                ?parent skos:notation ?parent_notation .
                ?parent rdf:value ?parent_value .
            }
        }
    """
    
    results = graph.query(query)
    
    requirements_list = []
    for row in results:
        req_text = str(row.value) if row.value else ""
        if req_text.strip():
            requirements_list.append({
                'id': str(row.notation),
                'label': str(row.label) if row.label else "N/A",
                'text': req_text,
                'parent_id': str(row.parent_notation) if row.parent_notation else None,
                'parent_text': str(row.parent_value) if row.parent_value else None,
            })

    logging.info(f"Successfully extracted {len(requirements_list)} requirements with parent context.")
    return requirements_list

In [ ]:
def analyze_batch_with_llm(batch: list) -> list:
    """
    Sends a batch of requirements to the LLM for ISO/IEC/IEEE 29148 quality analysis.
    This version includes parent requirement text for better context.
    """
    if not client:
        logging.error("API client is not initialized. Skipping analysis.")
        return []

    system_prompt = """
    U bent een expert in requirements engineering en systems engineering, gespecialiseerd in het analyseren van eisen volgens de ISO/IEC/IEEE 29148 standaard.

    Uw taak is om een lijst met eisen te analyseren op basis van de kwaliteitscriteria zoals gedefinieerd in ISO/IEC/IEEE 29148.
    Voor elke eis krijgt u de specifieke tekst en, indien beschikbaar, de tekst van de overkoepelende 'oudereis' voor context.
    Baseer uw analyse op de specifieke eis, maar gebruik de oudereis om de relevantie en het doel beter te begrijpen.

    Evalueer elke eis op de volgende ISO/IEC/IEEE 29148 kwaliteitscriteria:

    1. NECESSARY (Noodzakelijk): Is de eis noodzakelijk en voegt deze waarde toe aan het systeem?
    2. UNAMBIGUOUS (Eenduidig): Is de eis duidelijk geformuleerd zonder ruimte voor meerdere interpretaties?
    3. COMPLETE (Compleet): Bevat de eis alle noodzakelijke informatie zonder TBD's of open punten?
    4. SINGULAR (Enkelvoudig): Beschrijft de eis slechts één specifieke eis (geen 'en/of' constructies)?
    5. FEASIBLE (Haalbaar): Is de eis technisch en economisch realiseerbaar binnen de context?
    6. VERIFIABLE (Verifieerbaar): Kan de eis objectief getest of geverifieerd worden?
    7. TRACEABLE (Traceerbaar): Is de eis identificeerbaar en traceerbaar?
    8. IMPLEMENTATION_FREE (Implementatie-onafhankelijk): Beschrijft de eis WAT er nodig is, niet HOE het moet worden geïmplementeerd?

    De input is in het Nederlands en uw volledige output, inclusief alle rechtvaardigingen en suggesties, moet ook in het Nederlands zijn.

    Uw antwoord MOET een enkel, geldig JSON-object zijn dat één sleutel bevat, "results", die een lijst met woordenboeken bevat.
    Elk woordenboek moet overeenkomen met een input-eis en de volgende Engelse sleutels bevatten:
    - "id": De identificatiecode van de specifieke eis.
    - "necessary": Een boolean (true/false).
    - "necessary_justification": Een string in het Nederlands die uw beslissing uitlegt.
    - "unambiguous": Een boolean (true/false).
    - "unambiguous_justification": Een string in het Nederlands die uw beslissing uitlegt.
    - "complete": Een boolean (true/false).
    - "complete_justification": Een string in het Nederlands die uw beslissing uitlegt.
    - "singular": Een boolean (true/false).
    - "singular_justification": Een string in het Nederlands die uw beslissing uitlegt.
    - "feasible": Een boolean (true/false).
    - "feasible_justification": Een string in het Nederlands die uw beslissing uitlegt.
    - "verifiable": Een boolean (true/false).
    - "verifiable_justification": Een string in het Nederlands die uw beslissing uitlegt.
    - "traceable": Een boolean (true/false).
    - "traceable_justification": Een string in het Nederlands die uw beslissing uitlegt.
    - "implementation_free": Een boolean (true/false).
    - "implementation_free_justification": Een string in het Nederlands die uw beslissing uitlegt.
    - "quality_score": Een geheel getal van 0 tot 8 (aantal criteria dat wordt voldaan).
    - "suggestion": Een string in het Nederlands met een concreet voorstel om de eis te verbeteren volgens ISO/IEC/IEEE 29148.

    Lever geen tekst of uitleg buiten deze JSON-structuur.
    """

    user_prompt = json.dumps(batch, ensure_ascii=False, indent=2)

    try:
        logging.info(f"Sending a batch of {len(batch)} requirements for ISO/IEC/IEEE 29148 analysis...")
        
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_completion_tokens=MAX_COMPLETION_TOKENS,
            response_format={"type": "json_object"}
        )
        
        response_content = response.choices[0].message.content
        results_json = json.loads(response_content)
        
        if "results" in results_json and isinstance(results_json["results"], list):
            batch_results = results_json["results"]
            logging.info(f"Batch processed successfully. Received {len(batch_results)} analyses.")
            return batch_results
        else:
            logging.error("JSON response did not contain the expected 'results' list.")
            logging.error(f"Raw response received: {response_content}")
            return []

    except json.JSONDecodeError as json_err:
        logging.error(f"JSON decoding failed: {json_err}")
        logging.error(f"Raw response received: {response_content}")
        return []
    except Exception as e:
        logging.error(f"An error occurred during the OpenAI API call: {e}")
        return []

In [5]:
def process_single_batch(batch_data: tuple) -> list:
    """
    Processes a single batch of requirements, now including parent context in the prompt.
    """
    batch_index, batch = batch_data
    logging.info(f"Worker starting on batch {batch_index}...")
    
    original_req_map = {req['id']: req for req in batch}
    
    # --- CHANGE: Create a more detailed prompt object ---
    prompt_batch = []
    for req in batch:
        prompt_item = {"id": req["id"], "text": req["text"]}
        if req["parent_text"]:
            prompt_item["parent_context"] = req["parent_text"]
        prompt_batch.append(prompt_item)
    # --- END OF CHANGE ---
    
    batch_results = analyze_batch_with_llm(prompt_batch)
    
    processed_analyses = []
    if batch_results:
        for analysis in batch_results:
            req_id = analysis.get('id')
            original_req = original_req_map.get(req_id)
            if original_req:
                analysis['label'] = original_req['label']
                analysis['full_text'] = original_req['text']
                # Add parent info to the final output for traceability
                analysis['parent_id'] = original_req['parent_id']
                processed_analyses.append(analysis)
            else:
                logging.warning(f"Received analysis for an unknown ID '{req_id}' in batch {batch_index}.")
    else:
        logging.warning(f"Batch {batch_index} returned no valid results.")
        for req in batch:
            processed_analyses.append({
                'id': req['id'], 
                'label': req['label'], 
                'full_text': req['text'],
                'parent_id': req['parent_id'],
                'suggestion': 'ERROR: Failed to get analysis from LLM.'
            })
            
    return processed_analyses

In [6]:
# import concurrent.futures
# from tqdm import tqdm

# def main_concurrent():
#     """
#     Orchestrates the entire SMART requirement analysis workflow using concurrent processing
#     to maximize API throughput.
#     """
#     logging.info("--- STARTING CONCURRENT SMART ANALYSIS SCRIPT ---")

#     # Step 1: Extract requirements from the source file
#     requirements = extract_requirements_from_file(INPUT_FILE_PATH)

#     if not requirements:
#         logging.error("No requirements found to analyze. Stopping script.")
#         return

#     # --- NEW: Apply the analysis limit for testing ---
#     if MAX_REQUIREMENTS_TO_ANALYZE is not None and MAX_REQUIREMENTS_TO_ANALYZE > 0:
#         logging.info(f"TEST MODE: Limiting analysis to the first {MAX_REQUIREMENTS_TO_ANALYZE} requirements.")
#         requirements = requirements[:MAX_REQUIREMENTS_TO_ANALYZE]
#     # --- END OF NEW CODE ---

#     # Step 2: Create all batches in advance
#     batches = [
#         (i // BATCH_SIZE + 1, requirements[i:i + BATCH_SIZE]) 
#         for i in range(0, len(requirements), BATCH_SIZE)
#     ]
    
#     all_analyses = []
    
#     # --- Concurrent Processing ---
#     # Set max_workers to a number that approaches your RPM limit.
#     # 15 is a safe and fast starting point.
#     CONCURRENT_WORKERS = 15 
    
#     logging.info(f"Starting concurrent processing with {CONCURRENT_WORKERS} workers for {len(requirements)} requirements.")
    
#     with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENT_WORKERS) as executor:
#         # Use tqdm to create a progress bar for our batches
#         results_iterator = list(tqdm(executor.map(process_single_batch, batches), total=len(batches)))

#     # Flatten the list of lists returned by the executor
#     for result_list in results_iterator:
#         all_analyses.extend(result_list)

#     # Step 3: Create DataFrame and save to Excel
#     if all_analyses:
#         results_df = pd.DataFrame(all_analyses)
        
#         # Reorder columns for better readability
#         ordered_columns = [
#             'id', 'label', 'smart_score',
#             'specific', 'specific_justification',
#             'measurable', 'measurable_justification',
#             'achievable', 'achievable_justification',
#             'relevant', 'relevant_justification',
#             'time_bound', 'time_bound_justification',
#             'suggestion', 'full_text'
#         ]
#         for col in ordered_columns:
#             if col not in results_df.columns:
#                 results_df[col] = None
        
#         results_df = results_df[ordered_columns]

#         try:
#             results_df.to_excel(OUTPUT_FILE_PATH, index=False, engine='openpyxl')
#             logging.info(f"Analysis complete. Results for {len(results_df)} requirements saved to: {OUTPUT_FILE_PATH}")
#         except Exception as e:
#             logging.error(f"Failed to save Excel file: {e}")
#     else:
#         logging.warning("No analysis results were generated to save.")

#     logging.info("--- SCRIPT FINISHED ---")


# # --- Run the script ---
# # Make sure to install tqdm for the progress bar: pip install tqdm
# if __name__ == "__main__":
#     main_concurrent()

In [ ]:
import concurrent.futures
from tqdm import tqdm
import pandas as pd

def main_concurrent():
    """
    Orchestrates the entire ISO/IEC/IEEE 29148 requirement analysis workflow, producing a two-sheet
    Excel report: a detailed analysis and a parent summary.
    """
    logging.info("--- STARTING CONCURRENT ISO/IEC/IEEE 29148 ANALYSIS SCRIPT ---")

    # --- PHASE 1: PARSE RDF to get contextual data ---
    all_requirements = extract_requirements_from_file(INPUT_FILE_PATH)

    if not all_requirements:
        logging.error("No requirements found after parsing. Stopping script.")
        return

    # Apply the analysis limit for testing if set
    requirements_to_process = all_requirements
    if MAX_REQUIREMENTS_TO_ANALYZE is not None and MAX_REQUIREMENTS_TO_ANALYZE > 0:
        logging.info(f"TEST MODE: Limiting analysis to the first {MAX_REQUIREMENTS_TO_ANALYZE} requirements.")
        requirements_to_process = all_requirements[:MAX_REQUIREMENTS_TO_ANALYZE]
    
    # --- PHASE 2: ANALYZE DATA IN PARALLEL ---
    batches = [
        (i // BATCH_SIZE + 1, requirements_to_process[i:i + BATCH_SIZE]) 
        for i in range(0, len(requirements_to_process), BATCH_SIZE)
    ]
    
    all_analyses = []
    CONCURRENT_WORKERS = 15
    logging.info(f"Starting analysis with {CONCURRENT_WORKERS} workers for {len(requirements_to_process)} requirements.")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENT_WORKERS) as executor:
        results_iterator = list(tqdm(executor.map(process_single_batch, batches), total=len(batches)))

    for result_list in results_iterator:
        all_analyses.extend(result_list)

    # --- PHASE 3: PROCESS RESULTS AND CREATE EXCEL FILE ---
    if not all_analyses:
        logging.warning("No analysis results were generated to save.")
        return

    # Create the detailed DataFrame first with ISO/IEC/IEEE 29148 columns
    detailed_df = pd.DataFrame(all_analyses)
    ordered_columns = [
        'id', 'label', 'quality_score', 'parent_id',
        'necessary', 'necessary_justification',
        'unambiguous', 'unambiguous_justification',
        'complete', 'complete_justification',
        'singular', 'singular_justification',
        'feasible', 'feasible_justification',
        'verifiable', 'verifiable_justification',
        'traceable', 'traceable_justification',
        'implementation_free', 'implementation_free_justification',
        'suggestion', 'full_text'
    ]
    for col in ordered_columns:
        if col not in detailed_df.columns:
            detailed_df[col] = None
    detailed_df = detailed_df[ordered_columns]

    # --- Create the Parent Summary DataFrame ---
    logging.info("Generating parent summary report...")
    # Filter for requirements that actually have a parent
    children_df = detailed_df[detailed_df['parent_id'].notna()].copy()
    
    # Safely convert quality_score to numeric, coercing errors to NaN
    children_df['quality_score'] = pd.to_numeric(children_df['quality_score'], errors='coerce')

    if not children_df.empty:
        summary = children_df.groupby('parent_id').agg(
            child_req_ids=('id', lambda x: ', '.join(x)),
            number_of_children=('id', 'count'),
            average_quality_score=('quality_score', 'mean')
        ).reset_index()

        # Get the parent requirement text from the original list
        parent_info = {req['id']: req['text'] for req in all_requirements}
        summary['parent_text'] = summary['parent_id'].map(parent_info)
        
        # Reorder columns for the summary sheet
        summary = summary[['parent_id', 'parent_text', 'number_of_children', 'average_quality_score', 'child_req_ids']]
    else:
        # Create an empty summary if there are no parent-child relationships
        summary = pd.DataFrame(columns=['parent_id', 'parent_text', 'number_of_children', 'average_quality_score', 'child_req_ids'])

    # --- Save both DataFrames to a single Excel file with two sheets ---
    try:
        with pd.ExcelWriter(OUTPUT_FILE_PATH, engine='openpyxl') as writer:
            detailed_df.to_excel(writer, sheet_name='ISO 29148 Analysis', index=False)
            summary.to_excel(writer, sheet_name='Parent Summary', index=False)
        logging.info(f"Analysis complete. Two-sheet report saved to: {OUTPUT_FILE_PATH}")
    except Exception as e:
        logging.error(f"Failed to save Excel file: {e}")

    logging.info("--- SCRIPT FINISHED ---")


# --- Run the script ---
if __name__ == "__main__":
    main_concurrent()